## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [ ]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################

        # 万用近似原理: 一个具有线性输出层和至少一个使用"挤压"性质的激活函数的隐藏层, 只要隐藏层神经单元数量足够,
        # 就可以拟合实数空间中的有界闭集函数
        initializer = tf.keras.initializers.GlorotNormal()
        self.W1 = tf.Variable(initializer([28*28, 100]), dtype=tf.float32)
        self.b1 = tf.Variable(tf.zeros([100]), dtype=tf.float32)
        self.W2 = tf.Variable(initializer([100, 10]), dtype=tf.float32)
        self.b2 = tf.Variable(tf.zeros([10]), dtype=tf.float32)

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 28*28])
        
        h1 = tf.matmul(x, self.W1) + self.b1
        h1_relu = tf.nn.relu(h1)
        
        logits = tf.matmul(h1_relu, self.W2) + self.b2
        
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
for epoch in range(500):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.3505526 ; accuracy 0.12725
epoch 1 : loss 2.3405266 ; accuracy 0.13206667
epoch 2 : loss 2.3306808 ; accuracy 0.13611667
epoch 3 : loss 2.3210013 ; accuracy 0.14125
epoch 4 : loss 2.3114746 ; accuracy 0.14628333
epoch 5 : loss 2.3020868 ; accuracy 0.15098333
epoch 6 : loss 2.2928276 ; accuracy 0.1562
epoch 7 : loss 2.2836926 ; accuracy 0.16136667
epoch 8 : loss 2.2746713 ; accuracy 0.16593333
epoch 9 : loss 2.2657537 ; accuracy 0.17106667
epoch 10 : loss 2.256935 ; accuracy 0.1755
epoch 11 : loss 2.2482088 ; accuracy 0.1797
epoch 12 : loss 2.2395651 ; accuracy 0.1845
epoch 13 : loss 2.2310011 ; accuracy 0.18888333
epoch 14 : loss 2.2225134 ; accuracy 0.1939
epoch 15 : loss 2.2140985 ; accuracy 0.19893333
epoch 16 : loss 2.2057517 ; accuracy 0.20365
epoch 17 : loss 2.197466 ; accuracy 0.20801666
epoch 18 : loss 2.189241 ; accuracy 0.21293333
epoch 19 : loss 2.1810725 ; accuracy 0.21788333
epoch 20 : loss 2.1729577 ; accuracy 0.22253333
epoch 21 : loss 2.1648939 ; accura